<a href="https://colab.research.google.com/github/birkancekic/telecom-churn-revenue-retention-cockpit/blob/main/telecom_churn_dataset_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_records = 20000

customer_id = np.arange(100001, 100001 + n_records)
tenure_months = np.random.randint(1, 73, size=n_records)  # 1 - 72 ay arası müşteri yaşı
contract_type = np.random.choice(
    ["Month-to-Month", "One Year", "Two Year"],
    size=n_records,
    p=[0.55, 0.25, 0.20]
)
payment_method = np.random.choice(
    ["Electronic Check", "Mailed Check", "Bank Transfer", "Credit Card"],
    size=n_records
)
monthly_charges = np.round(np.random.uniform(20.0, 130.0, size=n_records), 2)
support_tickets = np.random.poisson(lam=1.6, size=n_records)

# Gerçekçi churn olasılığı (Lojistik fonksiyon)
# Kısa sözleşme, yüksek fatura ve çok destek talebi churn riskini katlar
linear_component = (
    -1.4
    - 0.045 * tenure_months
    + 0.75 * (contract_type == "Month-to-Month")
    - 0.50 * (contract_type == "Two Year")
    + 0.40 * support_tickets
    + 0.012 * (monthly_charges - 60)
)
churn_prob = 1 / (1 + np.exp(-linear_component))
churn_flag = np.random.binomial(1, churn_prob)

churn_status = np.where(churn_flag == 1, "Churn", "Active")

# Risk Segmentasyonu (Tıpkı bankacılık fazları gibi)
risk_segment = np.where(
    churn_prob >= 0.65, "Yüksek Risk",
    np.where(churn_prob >= 0.35, "Orta Risk", "Düşük Risk")
)

# Aylık risk altındaki gelir (MRR at Risk)
monthly_risk_exposure = np.round(monthly_charges * churn_prob, 2)

df_churn = pd.DataFrame({
    "customer_id": customer_id,
    "tenure_months": tenure_months,
    "contract_type": contract_type,
    "payment_method": payment_method,
    "monthly_charges": monthly_charges,
    "support_tickets": support_tickets,
    "churn_probability": np.round(churn_prob, 3),
    "churn_status": churn_status,
    "risk_segment": risk_segment,
    "monthly_risk_exposure": monthly_risk_exposure
})

df_churn.to_csv("telecom_churn_dataset.csv", index=False)
print("Veri seti oluşturuldu: 20,000 satır | telecom_churn_dataset.csv")

Veri seti oluşturuldu: 20,000 satır | telecom_churn_dataset.csv
